In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.sans-serif'] = "Arial"
# Then, "ALWAYS use sans-serif fonts"
matplotlib.rcParams['font.family'] = "sans-serif"
matplotlib.rcParams['font.sans-serif'] = "Arial"
# Then, "ALWAYS use sans-serif fonts"
matplotlib.rcParams['font.family'] = "sans-serif"
import numpy as np
import mygene


In [2]:
## Developed by Ruilin Tian, Kampmann lab, UCSF; 01/03/2019
## Requirement:
#   MAGeCK--can be downloaded at https://sourceforge.net/p/mageck/wiki/Home/
#   Python packages: pandas, numpy, scipy
# Usage: python MAGeCK-iNC_v5.2.py
# The script takes the counts file as input and generates following output files:
#   1. MAGeck outputs: a PDF summary, sgRNA_summary.txt, gene_summary.txt   
#   2. U test outputs: phenotypes-pvalues for all genes (_all_genes.csv), phenotypes-pvalues for FDR hits (_hits.csv), a volcano plot

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import mannwhitneyu,ttest_ind
import random
from random import shuffle
import subprocess
import shlex
import sys

def execute(command):
    print (command)
    subprocess.call(shlex.split(command))

def merge_counts(input_files,sample_names):
    count_files =  input_files.split(',')
    sample_names = sample_names.split(',')
    dfs = []
    for i,count_file in enumerate(count_files):
        df = pd.read_table(count_file,names =['sgRNA',sample_names[i]])
        dfs.append(df)
    df_merge = reduce(lambda x, y: pd.merge(x, y, on = 'sgRNA'), dfs).fillna(0.0)
    df_merge['Gene'] =df_merge['sgRNA'].apply(lambda x:x.split('_')[0] if (x.split('_')[2].endswith('-P1P2')) else  x.split('_')[0] + '_' + x.split('_')[2].split('-')[-1] )
    cols = df_merge.columns.tolist()
    cols = [cols[0]]+[cols[-1]] + cols[1:-1]
    df_merge = df_merge[cols]
    return df_merge

def negative_control_sgRNAs(df):
    return df[df['Gene'].str.contains('non-targeting')]['sgRNA'].tolist()


def product_threshold_fdr(df,fdr = 0.05):
    maxi = abs(df['product']).max()
    for pro in np.arange(0,maxi,0.1):
        df_thres = df[abs(df['product'])>pro]
        if (1.0 * len(df_thres[df_thres['index'].str.contains('NTC',na=False)]) / len(df_thres)) < fdr:
            break
    return pro, df_thres

def rank_test(df):
    df_ntc = df[df['Gene'].str.contains('non-targeting')]
    df_targeting =  df[~df['Gene'].str.contains('non-targeting')]  
    ntc_sgRNA_p = list(df_ntc['score'])
    ntc_sgRNA_p_lfc = zip(list(df_ntc['score']),list(df_ntc['LFC']))
    genes = df_targeting['Gene'].unique()
    num_of_genes = len(genes)
    gene_lfc_p = {}
    for gene in genes:
        df_gene = df_targeting[df_targeting['Gene'] == gene].sort_values('score', ascending = False)
        lfc = df_gene.iloc[:]['LFC'].mean()
        x, pvalue = mannwhitneyu(list(df_gene['score'][:]),ntc_sgRNA_p,alternative='two-sided')
        
        gene_lfc_p[gene] = [lfc,pvalue]

    random.seed(10)
    for j in range(num_of_genes):
        shuffle(ntc_sgRNA_p_lfc)
        ntc_selected = ntc_sgRNA_p_lfc[:5]
        ntc_selected_p = [i[0] for i in ntc_selected]
        ntc_lfc = np.mean([i[1] for i in sorted(ntc_selected, key=lambda x: x[0])][:])
        x, ntc_pvalue = mannwhitneyu(ntc_selected_p[:],ntc_sgRNA_p,alternative='two-sided')
        gene_lfc_p['NTC_' + str(j)] = [ntc_lfc, ntc_pvalue]
    return gene_lfc_p


In [3]:
mapping = pd.read_csv('../related_files/mouse_gene_name_old_new_mapping.csv')
mapping = dict(zip(mapping['query'], mapping['symbol']))

In [ ]:
import os
mageck_path = '../new_mageck/'
output_path = '../new_iNC/'
# for sample in os.listdir(mageck_path):
sample = 'Syn1-GW-all_4M'
mageck_file = mageck_path+sample+'/'+sample+'.sgrna_summary.txt'
if os.path.exists(mageck_file): 

    if sample not in os.listdir(output_path):
        output = output_path + sample
        os.makedirs(output)

    df_mageck = pd.read_table(mageck_file)
    df = pd.DataFrame(rank_test(df_mageck)).T
    df.columns = ['epsilon','pvalue']
    df.reset_index(inplace=True)
    df['gene'] = df['index'].apply(lambda x:x.split('_')[0])
    df['gene'] = [mapping[i] if i in mapping.keys() else i for i in df['gene'].tolist()]
    df_ntc = df[df['gene']=='NTC']
    df['epsilon'] = df['epsilon'] - df_ntc['epsilon'].median()
    df['product'] = df['epsilon'] * (-np.log10(df['pvalue']))
    df.sort_values('product',ascending=False)
    fdr=0.1
    # FDR
    thres, df_hits = product_threshold_fdr(df,fdr)
    df_pos = df_hits[df_hits['epsilon']>0]
    df_neg = df_hits[df_hits['epsilon']<0]
    # df.sort_values('product',ascending=False).to_csv(out_put + '/' + screen_name + '_all_genes.csv',index=False)
    # df_hits.sort_values('product',ascending=False).to_csv(out_put + '/' + screen_name + '_fdr%s_product%.3f_hits.csv'%(fdr,thres),index=False)
    df_ntc = df[df['index'].str.contains('NTC')]
    ########### volcano plot
    df['hits'] = 0
    df.loc[df['index'].isin(df_pos['index']),'hits'] = 1
    df.loc[df['index'].isin(df_neg['index']),'hits'] = -1
    df.sort_values('product',ascending=True).to_csv(output_path + sample + "/" + sample + '_all_genes.csv',index=False)


/Users/Ruilin/miniconda2/lib/python2.7/site-packages/ipykernel_launcher.py:14: FutureWarning: read_table is deprecated, use read_csv instead, passing sep='\t'.
  
